In [ ]:
import os
import numpy as np
import pandas as pd

import sys
sys.path.insert(0, "../../packages/ProMERGE_utils")
from promerge_utils import ppi_from_emb, concat_emb

### Permute

In [ ]:
os.makedirs("../../result/Hela_permute/input/", exist_ok=True)

In [ ]:
for f in [
    "SEC-Interphase.tsv",
    "SEC-Mitosis.tsv",
    "TPP-Interphase.tsv",
    "TPP-Mitosis.tsv"
]:
    df = pd.read_csv(
        f"../../result/Hela/input/{f}",
        sep="\t", index_col=0
    )
    df.index = np.random.permutation(df.index)
    df = df.sort_index()
    df.to_csv(
        f"../../result/Hela_permute/input/{f}", sep="\t"
    )

### Concatenate the modality embeddings

In [ ]:
df1 = pd.read_csv("../../result/Hela_permute/input/SEC-Interphase.tsv", sep="\t", index_col=0)
df2 = pd.read_csv("../../result/Hela_permute/input/TPP-Interphase.tsv", sep="\t", index_col=0)
df_concat = concat_emb([df1, df2])
df_concat.to_csv("../../result/Hela_permute/input/Concat-Interphase.tsv", sep="\t", header=True, index=True)

In [ ]:
df1 = pd.read_csv("../../result/Hela_permute/input/SEC-Mitosis.tsv", sep="\t", index_col=0)
df2 = pd.read_csv("../../result/Hela_permute/input/TPP-Mitosis.tsv", sep="\t", index_col=0)
df_concat = concat_emb([df1, df2])
df_concat.to_csv("../../result/Hela_permute/input/Concat-Mitosis.tsv", sep="\t", header=True, index=True)

### Stats for future filtering

In [ ]:
# load genes in each experiment
sec_interphase = set(pd.read_csv("../../result/Hela_permute/input/SEC-Interphase.tsv", sep="\t", index_col=0).index)
sec_mitosis = set(pd.read_csv("../../result/Hela_permute/input/SEC-Mitosis.tsv", sep="\t", index_col=0).index)
tpp_interphase = set(pd.read_csv("../../result/Hela_permute/input/TPP-Interphase.tsv", sep="\t", index_col=0).index)
tpp_mitosis = set(pd.read_csv("../../result/Hela_permute/input/TPP-Mitosis.tsv", sep="\t", index_col=0).index)

In [ ]:
# total possible pairs in each cell line
interphase_genes = sec_interphase | tpp_interphase
interphase_pairs = len(interphase_genes) * (len(interphase_genes)-1) // 2

mitosis_genes = sec_mitosis | tpp_mitosis
mitosis_pairs = len(mitosis_genes) * (len(mitosis_genes)-1) // 2

In [ ]:
# top 1%, add other when needed
stat = pd.DataFrame(
    [
        [int(interphase_pairs*0.01), int(mitosis_pairs*0.01)]
    ],
    index=["Percent1"],
    columns=["Interphase", "Mitosis"]
)
stat.to_csv("../../result/Hela_permute/input/pair_stat.tsv", sep="\t")